# # HW08-09: PyTorch MLP - Регуляризация и оптимизация обучения

# ## 1. Импорты, seed и устройство

In [ ]:

# %%
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
from pathlib import Path
import time

# Настраиваем пути для сохранения артефактов
ARTIFACTS_DIR = Path('./artifacts')
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Артефакты будут сохранены в: {ARTIFACTS_DIR.absolute()}")

# %%
# Фиксируем seed для воспроизводимости
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    # Если есть CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Для детерминированных вычислений на CUDA (может замедлить обучение)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Seed установлен: {SEED}")

# %%
# Определяем устройство (GPU если доступно)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# ## 2. Данные и DataLoader

In [ ]:
# Загружаем EMNIST Balanced, создаем train/val split и DataLoader'ы.

# %%
# Определяем трансформации: преобразуем в тензор и нормализуем
# Значения 0.5 и 0.5 приводят пиксели из диапазона [0,1] в [-1,1]
# Это часто помогает обучению
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # для одного канала (grayscale)
])

# %%
# Загружаем тренировочную и тестовую части EMNIST Balanced
# Важно: EMNIST возвращает images (28x28) и labels (0-46 для balanced split)
train_dataset_full = torchvision.datasets.EMNIST(
    root='./data', split='balanced', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.EMNIST(
    root='./data', split='balanced', train=False, download=True, transform=transform
)

print(f"Размер тренировочного датасета (полный): {len(train_dataset_full)}")
print(f"Размер тестового датасета: {len(test_dataset)}")
print(f"Количество классов: {len(train_dataset_full.classes)}")
print(f"Форма изображения: {train_dataset_full[0][0].shape}")

# %%
# Разделяем тренировочную часть на train (80%) и val (20%)
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

# Используем random_split с фиксированным seed для воспроизводимости
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

print(f"Размер тренировочной выборки: {len(train_dataset)}")
print(f"Размер валидационной выборки: {len(val_dataset)}")

# %%
# Создаем DataLoader'ы
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoader'ы созданы с batch_size = {BATCH_SIZE}")

# %%
# Sanity check: проверяем shapes и диапазоны
for images, labels in train_loader:
    print(f"Shape батча изображений: {images.shape}")  # [batch, channels, height, width]
    print(f"Shape батча меток: {labels.shape}")
    print(f"Диапазон значений пикселей: [{images.min():.2f}, {images.max():.2f}]")
    print(f"Уникальные метки: {torch.unique(labels)}")
    print(f"Количество уникальных меток в батче: {len(torch.unique(labels))}")
    break  # только первый батч

# ## 3. Модель MLP

In [ ]:
# Создаем гибкую MLP модель для EMNIST (47 классов).

# %%
class MLP(nn.Module):
    """
    Гибкая MLP модель для классификации изображений.
    
    Args:
        input_size: размер входного вектора (для EMNIST 28*28 = 784)
        hidden_sizes: список размеров скрытых слоев
        num_classes: количество классов для выхода (для EMNIST Balanced = 47)
        activation: функция активации ('relu' или 'tanh')
        use_batchnorm: использовать ли BatchNorm после каждого линейного слоя (кроме последнего)
        dropout_rate: вероятность dropout'а (0 = не использовать)
    """
    def __init__(self, input_size=784, hidden_sizes=[256, 128], num_classes=47,
                 activation='relu', use_batchnorm=False, dropout_rate=0.0):
        super().__init__()
        
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.num_classes = num_classes
        self.use_batchnorm = use_batchnorm
        self.dropout_rate = dropout_rate
        
        # Выбираем функцию активации
        if activation == 'relu':
            self.activation_fn = nn.ReLU()
        elif activation == 'tanh':
            self.activation_fn = nn.Tanh()
        else:
            raise ValueError(f"Неподдерживаемая активация: {activation}")
        
        # Строим слои
        layers = []
        
        # Входной слой: Flatten + Linear
        layers.append(nn.Flatten())
        
        # Скрытые слои
        prev_size = input_size
        for i, hidden_size in enumerate(hidden_sizes):
            # Линейный слой
            layers.append(nn.Linear(prev_size, hidden_size))
            
            # BatchNorm (после Linear, но перед активацией)
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            # Активация
            layers.append(self.activation_fn)
            
            # Dropout (после активации)
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            
            prev_size = hidden_size
        
        # Выходной слой (logits) - 47 классов для EMNIST
        layers.append(nn.Linear(prev_size, num_classes))
        
        # Объединяем все в последовательную модель
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)
    
    def get_summary(self):
        """Возвращает краткое описание архитектуры для runs.csv"""
        bn_str = "BN" if self.use_batchnorm else "noBN"
        dropout_str = f"DO{self.dropout_rate}" if self.dropout_rate > 0 else "noDO"
        return f"MLP_{self.hidden_sizes}_{bn_str}_{dropout_str}"

# %%
# Тестируем модель на одном батче
test_model = MLP(input_size=784, hidden_sizes=[256, 128], num_classes=47)
test_model.to(device)

for images, labels in train_loader:
    images = images.to(device)
    output = test_model(images)
    print(f"Выход модели shape: {output.shape}")  # [batch_size, 47]
    print(f"Логиты (пример): {output[0, :5]}...")  # первые 5 значений
    break

# Очищаем тестовую модель
del test_model

# ## 4. Функции для обучения и оценки

In [ ]:

# %%
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Обучает модель одну эпоху.
    Возвращает средний loss и accuracy за эпоху.
    """
    model.train()  # включаем тренировочный режим (важно для Dropout/BatchNorm)
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Обнуляем градиенты
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass и оптимизация
        loss.backward()
        optimizer.step()
        
        # Статистика
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# %%
def evaluate(model, data_loader, criterion, device):
    """
    Оценивает модель на датасете.
    Возвращает средний loss и accuracy.
    """
    model.eval()  # включаем режим оценки (важно для Dropout/BatchNorm)
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # отключаем вычисление градиентов
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# %%
def train_model(model, train_loader, val_loader, criterion, optimizer, 
                num_epochs, device, early_stopping_patience=None, experiment_name=""):
    """
    Полный цикл обучения с опциональным early stopping.
    
    Returns:
        history: словарь с историями loss/accuracy
        best_model_state: состояние модели с лучшей val_accuracy
        best_epoch: эпоха, на которой достигнут лучший результат
        early_stopped_epoch: эпоха, на которой сработал early stopping (или None)
    """
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_val_acc = 0.0
    best_model_state = None
    best_epoch = -1
    early_stopped_epoch = None
    
    # Для early stopping
    patience_counter = 0
    
    for epoch in range(num_epochs):
        # Обучение
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        
        # Валидация
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        # Сохраняем историю
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Проверяем, улучшилась ли метрика на валидации
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()  # копируем состояние
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping проверка
        if early_stopping_patience is not None and patience_counter >= early_stopping_patience:
            early_stopped_epoch = epoch
            print(f"  Early stopping сработал на эпохе {epoch+1}")
            break
        
        # Печатаем прогресс каждые 5 эпох
        if (epoch + 1) % 5 == 0:
            print(f"  Эпоха {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    return history, best_model_state, best_epoch, early_stopped_epoch

# ## 5. Функция для запуска экспериментов и логирования

In [ ]:

# %%
def run_experiment(experiment_id, model_config, train_config, 
                   train_loader, val_loader, test_loader, device, seed=SEED):
    """
    Запускает один эксперимент, сохраняет результаты и возвращает словарь с метриками.
    """
    print(f"\n=== Запуск эксперимента {experiment_id} ===")
    print(f"Конфиг модели: {model_config}")
    print(f"Конфиг обучения: {train_config}")
    
    # Устанавливаем seed для воспроизводимости
    set_seed(seed)
    
    # Создаем модель
    model = MLP(**model_config).to(device)
    
    # Создаем оптимизатор
    if train_config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=train_config['lr'], 
                               weight_decay=train_config.get('weight_decay', 0))
    elif train_config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=train_config['lr'],
                             momentum=train_config.get('momentum', 0),
                             weight_decay=train_config.get('weight_decay', 0))
    else:
        raise ValueError(f"Неизвестный оптимизатор: {train_config['optimizer']}")
    
    criterion = nn.CrossEntropyLoss()
    
    # Обучаем модель
    start_time = time.time()
    history, best_model_state, best_epoch, early_stopped_epoch = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=train_config['num_epochs'],
        device=device,
        early_stopping_patience=train_config.get('early_stopping_patience'),
        experiment_name=experiment_id
    )
    training_time = time.time() - start_time
    
    # Загружаем лучшую модель для финальной оценки
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # Оцениваем на валидации (еще раз, для уверенности)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Оцениваем на тесте (только для финальной модели, но для E4 мы сделаем это отдельно)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    # Формируем результаты
    results = {
        'experiment_id': experiment_id,
        'dataset': 'EMNIST_Balanced',
        'seed': seed,
        'model_summary': model.get_summary(),
        'optimizer': train_config['optimizer'],
        'lr': train_config['lr'],
        'momentum': train_config.get('momentum', ''),
        'weight_decay': train_config.get('weight_decay', 0),
        'epochs_trained': len(history['train_loss']),
        'best_val_accuracy': val_acc,
        'best_val_loss': val_loss,
        'test_accuracy': test_acc,
        'test_loss': test_loss,
        'best_epoch': best_epoch,
        'training_time': training_time,
        'early_stopped': early_stopped_epoch is not None,
        'early_stopped_epoch': early_stopped_epoch if early_stopped_epoch is not None else ''
    }
    
    # Для экспериментов с early stopping (E4) сохраняем модель
    if experiment_id == 'E4':
        torch.save(best_model_state, ARTIFACTS_DIR / 'best_model.pt')
        print(f"  Лучшая модель сохранена в {ARTIFACTS_DIR / 'best_model.pt'}")
        
        # Сохраняем конфиг лучшей модели
        best_config = {
            'model_config': model_config,
            'train_config': train_config,
            'seed': seed,
            'dataset': 'EMNIST_Balanced',
            'best_val_accuracy': val_acc,
            'best_epoch': best_epoch
        }
        with open(ARTIFACTS_DIR / 'best_config.json', 'w') as f:
            json.dump(best_config, f, indent=2)
        print(f"  Конфиг лучшей модели сохранен в {ARTIFACTS_DIR / 'best_config.json'}")
    
    # Сохраняем графики для лучшего эксперимента и для O1/O2
    if experiment_id == 'E4':
        plot_training_history(history, experiment_id, save=True, 
                              filename=FIGURES_DIR / 'curves_best.png')
    elif experiment_id in ['O1', 'O2']:
        # Для O1 и O2 мы сохраним общий график позже
        pass
    
    print(f"  Результаты: Val Acc={val_acc:.4f}, Test Acc={test_acc:.4f}, Время={training_time:.2f}с")
    
    return results, history

# %%
def plot_training_history(history, experiment_id, save=False, filename=None):
    """Рисует графики обучения"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # График loss
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{experiment_id}: Loss over epochs')
    ax1.legend()
    ax1.grid(True)
    
    # График accuracy
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{experiment_id}: Accuracy over epochs')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    
    if save and filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"  График сохранен в {filename}")
    
    plt.show()

# ## 6. Эксперименты Часть A (S08): Регуляризация (E1-E4)
# ### E1: Baseline (без Dropout и BatchNorm)

In [ ]:

# %%
# Общие параметры для всех экспериментов части A
BASE_CONFIG = {
    'input_size': 784,
    'hidden_sizes': [256, 128],  # 2 скрытых слоя
    'num_classes': 47,  # ВАЖНО: 47 классов для EMNIST Balanced
    'activation': 'relu'
}

TRAIN_CONFIG_BASE = {
    'optimizer': 'Adam',
    'lr': 0.001,  # стандартный lr для Adam
    'num_epochs': 30,
    'weight_decay': 0
}

# %%
# Список для хранения результатов всех экспериментов
all_results = []


# %%
model_config_e1 = {**BASE_CONFIG, 'use_batchnorm': False, 'dropout_rate': 0.0}
train_config_e1 = {**TRAIN_CONFIG_BASE}

results_e1, history_e1 = run_experiment(
    experiment_id='E1',
    model_config=model_config_e1,
    train_config=train_config_e1,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_e1)

# %%
# Смотрим графики E1
plot_training_history(history_e1, 'E1')


# %%
model_config_e2 = {**BASE_CONFIG, 'use_batchnorm': False, 'dropout_rate': 0.3}
train_config_e2 = {**TRAIN_CONFIG_BASE}

results_e2, history_e2 = run_experiment(
    experiment_id='E2',
    model_config=model_config_e2,
    train_config=train_config_e2,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_e2)

# %%
plot_training_history(history_e2, 'E2')


# %%
model_config_e3 = {**BASE_CONFIG, 'use_batchnorm': True, 'dropout_rate': 0.0}
train_config_e3 = {**TRAIN_CONFIG_BASE}

results_e3, history_e3 = run_experiment(
    experiment_id='E3',
    model_config=model_config_e3,
    train_config=train_config_e3,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_e3)

# %%
plot_training_history(history_e3, 'E3')


# Выбираем лучший по val_accuracy из E2 и E3

# %%
# Находим лучший эксперимент
e2_val_acc = results_e2['best_val_accuracy']
e3_val_acc = results_e3['best_val_accuracy']

print(f"Val Accuracy E2 (Dropout): {e2_val_acc:.4f}")
print(f"Val Accuracy E3 (BatchNorm): {e3_val_acc:.4f}")

if e2_val_acc >= e3_val_acc:
    print("Выбираем Dropout как основу для E4")
    best_model_config = model_config_e2
else:
    print("Выбираем BatchNorm как основу для E4")
    best_model_config = model_config_e3

# Настраиваем E4 с early stopping
model_config_e4 = best_model_config
train_config_e4 = {
    **TRAIN_CONFIG_BASE,
    'num_epochs': 30,
    'early_stopping_patience': 5  # терпение 5 эпох
}

results_e4, history_e4 = run_experiment(
    experiment_id='E4',
    model_config=model_config_e4,
    train_config=train_config_e4,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_e4)

# %%
plot_training_history(history_e4, 'E4')

# ## 7. Эксперименты Часть B (S09): LR и оптимизаторы (O1-O3)

In [ ]:
 # Используем ту же архитектуру, что и в E4

# %%
# Фиксированная архитектура как в лучшей модели E4
fixed_model_config = model_config_e4.copy()  # копируем конфиг из E4

# Словарь для хранения историй O1 и O2 для общего графика
lr_extremes_histories = {}


# %%
train_config_o1 = {
    'optimizer': 'Adam',
    'lr': 0.1,  # слишком большой
    'num_epochs': 8,  # достаточно мало эпох, чтобы увидеть проблему
    'weight_decay': 0
}

results_o1, history_o1 = run_experiment(
    experiment_id='O1',
    model_config=fixed_model_config,
    train_config=train_config_o1,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_o1)
lr_extremes_histories['O1'] = history_o1

# %%
plot_training_history(history_o1, 'O1')


# %%
train_config_o2 = {
    'optimizer': 'Adam',
    'lr': 1e-5,  # слишком маленький
    'num_epochs': 8,
    'weight_decay': 0
}

results_o2, history_o2 = run_experiment(
    experiment_id='O2',
    model_config=fixed_model_config,
    train_config=train_config_o2,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_o2)
lr_extremes_histories['O2'] = history_o2

# %%
plot_training_history(history_o2, 'O2')


# %%
train_config_o3 = {
    'optimizer': 'SGD',
    'lr': 0.01,  # разумный lr для SGD (обычно больше, чем для Adam)
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'num_epochs': 15
}

results_o3, history_o3 = run_experiment(
    experiment_id='O3',
    model_config=fixed_model_config,
    train_config=train_config_o3,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device
)
all_results.append(results_o3)

# %%
plot_training_history(history_o3, 'O3')

# ## 8. Создание общего графика для O1 и O2

In [ ]:

# %%
def plot_lr_extremes(histories_dict, save_path=None):
    """Создает график, показывающий влияние слишком большого и маленького LR"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # O1: Слишком большой LR
    ax = axes[0, 0]
    epochs_o1 = range(1, len(histories_dict['O1']['train_loss']) + 1)
    ax.plot(epochs_o1, histories_dict['O1']['train_loss'], 'b-', label='Train Loss', alpha=0.7)
    ax.plot(epochs_o1, histories_dict['O1']['val_loss'], 'r-', label='Val Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('O1: LR = 0.1 (слишком большой)')
    ax.legend()
    ax.grid(True)
    
    ax = axes[0, 1]
    ax.plot(epochs_o1, histories_dict['O1']['train_acc'], 'b-', label='Train Acc', alpha=0.7)
    ax.plot(epochs_o1, histories_dict['O1']['val_acc'], 'r-', label='Val Acc', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('O1: LR = 0.1 (слишком большой)')
    ax.legend()
    ax.grid(True)
    
    # O2: Слишком маленький LR
    ax = axes[1, 0]
    epochs_o2 = range(1, len(histories_dict['O2']['train_loss']) + 1)
    ax.plot(epochs_o2, histories_dict['O2']['train_loss'], 'b-', label='Train Loss', alpha=0.7)
    ax.plot(epochs_o2, histories_dict['O2']['val_loss'], 'r-', label='Val Loss', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('O2: LR = 1e-5 (слишком маленький)')
    ax.legend()
    ax.grid(True)
    
    ax = axes[1, 1]
    ax.plot(epochs_o2, histories_dict['O2']['train_acc'], 'b-', label='Train Acc', alpha=0.7)
    ax.plot(epochs_o2, histories_dict['O2']['val_acc'], 'r-', label='Val Acc', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('O2: LR = 1e-5 (слишком маленький)')
    ax.legend()
    ax.grid(True)
    
    plt.suptitle('Сравнение экстремальных learning rates', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"График сохранен в {save_path}")
    
    plt.show()

# %%
# Создаем и сохраняем график
plot_lr_extremes(lr_extremes_histories, save_path=FIGURES_DIR / 'curves_lr_extremes.png')

# ## 9. Сохранение всех результатов 

In [ ]:

# %%
# Создаем DataFrame из всех результатов
df_results = pd.DataFrame(all_results)

# Переупорядочиваем колонки для читаемости
column_order = ['experiment_id', 'dataset', 'seed', 'model_summary', 'optimizer', 
                'lr', 'momentum', 'weight_decay', 'epochs_trained', 
                'best_val_accuracy', 'best_val_loss', 'test_accuracy', 'test_loss',
                'best_epoch', 'training_time', 'early_stopped', 'early_stopped_epoch']
df_results = df_results[column_order]

# Сохраняем в CSV
csv_path = ARTIFACTS_DIR / 'runs.csv'
df_results.to_csv(csv_path, index=False)
print(f"Результаты сохранены в {csv_path}")

# Выводим таблицу для проверки
df_results


# %%
# Загружаем лучшую модель
best_model = MLP(**model_config_e4).to(device)
best_model.load_state_dict(torch.load(ARTIFACTS_DIR / 'best_model.pt', map_location=device))

# Оцениваем на тесте
criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)

print(f"\n=== Финальная оценка лучшей модели (E4) ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Эксперимент E4: лучшая val_accuracy = {results_e4['best_val_accuracy']:.4f}")
print(f"Разрыв train/val можно посмотреть на сохраненном графике curves_best.png")